# CellLM Experiment 0 — GPU ladder

Runs the complete eight-rung, three-seed capacity gate. Completed seed checkpoints are written to the submitter's mounted output directory so they survive the job.

In [1]:
%pip install git+https://github.com/tomieiro/libPyCelNN.git
%pip install -e . --no-deps

  Cloning https://github.com/tomieiro/libPyCelNN.git to /tmp/pip-req-build-k4a5kdac


  Running command git clone --filter=blob:none --quiet https://github.com/tomieiro/libPyCelNN.git /tmp/pip-req-build-k4a5kdac


  Resolved https://github.com/tomieiro/libPyCelNN.git to commit 1685319256e5f167ea1fef9d5f9e48a7e8b69578
  DEPRECATION: Setting PIP_CONSTRAINT will not affect build constraints in the future, pip 26.2 will enforce this behaviour change. A possible replacement is to specify build constraints using --build-constraint or PIP_BUILD_CONSTRAINT. To disable this warning without any build constraints set --use-feature=build-constraint or PIP_USE_FEATURE="build-constraint".
  Installing build dependencies ... 

-

 \

 |

... (18 outputs left out)

 done
  Building editable for celllm (pyproject.toml) ... 

-

 done
  Created wheel for celllm: filename=celllm-0.1.0-0.editable-py3-none-any.whl size=1272 sha256=ef90c5e805382d126da20ba30d45b09af0d8c6e08c4ab9b3e3d9357ca951f849
  Stored in directory: /tmp/pip-ephem-wheel-cache-42zz8vsr/wheels/b0/41/ab/4e04a8678980f3355548f1261923479fbf2a713a7393a7dea8
Successfully built celllm


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import importlib
import shutil
import sys

root = Path('/workspace')
outputs = root / 'outputs'
output_checkpoints = outputs / 'checkpoints'
output_checkpoints.mkdir(parents=True, exist_ok=True)

# Checkpoints copied back into the project before a resubmission are staged
# into the mounted output directory and skipped by the ladder runner.
local_checkpoints = root / 'checkpoints'
if local_checkpoints.exists():
    for checkpoint in local_checkpoints.glob('*.pt'):
        destination = output_checkpoints / checkpoint.name
        if not destination.exists():
            shutil.copy2(checkpoint, destination)

data_dir = root / 'data'
text8 = data_dir / 'text8'
if not text8.exists():
    shutil.unpack_archive(data_dir / 'text8.zip', data_dir)

torch = importlib.import_module('torch')
assert torch.cuda.is_available(), 'the submitted job did not receive a GPU'
print('GPU:', torch.cuda.get_device_name(0), flush=True)

command = [
    sys.executable, '-m', 'celllm.ladder',
    '--data', str(text8),
    '--steps', '20000',
    '--device', 'cuda',
    '--checkpoint-dir', str(output_checkpoints),
    '--parallel-seeds', '3',
    '--out', str(outputs / 'experiment-0.json'),
]
print(' '.join(command), flush=True)
__import__('subprocess').run(command, check=True)

GPU: NVIDIA A100-SXM4-80GB


/usr/bin/python -m celllm.ladder --data /workspace/data/text8 --steps 20000 --device cuda --checkpoint-dir /workspace/outputs/checkpoints --parallel-seeds 3 --out /workspace/outputs/experiment-0.json


{"hardware": {"host": "02e3ee29beeb", "torch": "2.12.0a0+5aff3928d8.nv26.05", "cuda": "13.2", "device": "NVIDIA A100-SXM4-80GB"}}


=== rung A ===


saved /workspace/outputs/checkpoints/rung-A-seed-42.pt
saved /workspace/outputs/checkpoints/rung-A-seed-1337.pt
saved /workspace/outputs/checkpoints/rung-A-seed-2024.pt


{"rung": "A", "bpc_mean": 2.7863494580572064, "bpc_std": 0.007982633825522917, "core": 7, "total": 3490, "flops": 1032192, "latency_ms": 33.44893082976341}
=== rung B ===


... (10 outputs left out)

saved /workspace/outputs/checkpoints/rung-G-seed-42.pt
saved /workspace/outputs/checkpoints/rung-G-seed-1337.pt
saved /workspace/outputs/checkpoints/rung-G-seed-2024.pt


{"rung": "G", "bpc_mean": 2.2801508809119113, "bpc_std": 0.009283306630188417, "core": 17280, "total": 20763, "flops": 34586624, "latency_ms": 21.041040308773518}
=== rung H ===


saved /workspace/outputs/checkpoints/rung-H-seed-42.pt
saved /workspace/outputs/checkpoints/rung-H-seed-1337.pt
saved /workspace/outputs/checkpoints/rung-H-seed-2024.pt
{"rung": "H", "bpc_mean": 1.8311957350465804, "bpc_std": 0.0010410308496676946, "core": 394240, "total": 397723, "flops": 25387008, "latency_ms": 0.6363475695252419}

rung       BPC      ±      core     total    MFLOPs       ms
------------------------------------------------------------
A        2.786  0.008         7     3,490       1.0    33.45
B        2.740  0.009       896     4,379       1.0    33.67
C        2.656  0.004     1,920     5,403       3.1    21.27
D        2.566  0.014     2,944     6,427       5.2    21.63
E        2.478  0.004     4,992     8,475       9.4    20.76
F        2.373  0.007     9,088    12,571      17.8    21.32
G        2.280  0.009    17,280    20,763      34.6    21.04
H        1.831  0.001   394,240   397,723      25.4     0.64


CompletedProcess(args=['/usr/bin/python', '-m', 'celllm.ladder', '--data', '/workspace/data/text8', '--steps', '20000', '--device', 'cuda', '--checkpoint-dir', '/workspace/outputs/checkpoints', '--parallel-seeds', '3', '--out', '/workspace/outputs/experiment-0.json'], returncode=0)